In [ ]:
#!/usr/bin/env python3
"""
DATA255 Lab 1 - ONNX -> MXQ conversion script
Compatible with:
- model_imagenet20.onnx
- Input size: 240x240
- ImageNet normalization
"""

import argparse
import os
from pathlib import Path

import numpy as np
from PIL import Image

# Lab NPU compiler library (must exist in lab environment)
from qubee import mxq_compile


IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)


def preprocess_image(img_path, img_size):
    """
    Returns NCHW float32 array with ImageNet normalization.
    Shape: (1, 3, img_size, img_size)
    """
    img = Image.open(img_path).convert("RGB")
    img = img.resize((img_size, img_size), Image.BILINEAR)

    arr = np.asarray(img).astype(np.float32) / 255.0
    arr = (arr - IMAGENET_MEAN) / IMAGENET_STD
    arr = np.transpose(arr, (2, 0, 1))  # HWC -> CHW
    arr = np.expand_dims(arr, axis=0)   # Add batch dimension

    return arr


def load_calibration_images(calib_dir, img_size, max_images=500):
    """
    Load calibration images from folder.
    """
    calib_arrays = []

    supported_ext = [".jpg", ".jpeg", ".png", ".bmp", ".JPG", ".JPEG"]

    for file in os.listdir(calib_dir):
        if any(file.endswith(ext) for ext in supported_ext):
            img_path = os.path.join(calib_dir, file)
            calib_arrays.append(preprocess_image(img_path, img_size))

            if len(calib_arrays) >= max_images:
                break

    if len(calib_arrays) == 0:
        raise RuntimeError("No calibration images found!")

    return calib_arrays


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--onnx", required=True, help="Path to ONNX model")
    parser.add_argument("--mxq", required=True, help="Output MXQ path")
    parser.add_argument("--calib_dir", required=True, help="Calibration image folder")
    parser.add_argument("--img_size", type=int, default=240, help="Model input size")
    args = parser.parse_args()

    if not os.path.exists(args.onnx):
        raise FileNotFoundError("ONNX model not found!")

    if not os.path.exists(args.calib_dir):
        raise FileNotFoundError("Calibration folder not found!")

    print("Loading calibration images...")
    calib_data = load_calibration_images(args.calib_dir, args.img_size)

    print(f"Using {len(calib_data)} calibration images")
    print("Compiling ONNX -> MXQ ...")

   mxq_compile(
    model=args.onnx,
    save_path=args.mxq,


    calib_data_path=args.calib_dir,


    input_shapes=[[1, 3, args.img_size, args.img_size]],

    verbose=True,
)

    if os.path.exists(args.mxq):
        size_mb = os.path.getsize(args.mxq) / (1024 * 1024)
        print(f"\n MXQ generated successfully: {args.mxq}")
        print(f"File size: {size_mb:.2f} MB")
    else:
        print(" MXQ generation failed!")


if __name__ == "__main__":
    main()

In [ ]:
import argparse
import os
from pathlib import Path

import numpy as np
from PIL import Image

# Lab NPU compiler library (must exist in lab environment)
from qubee import mxq_compile


IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)


def preprocess_image(img_path, img_size):
    """
    Returns NCHW float32 array with ImageNet normalization.
    Shape: (1, 3, img_size, img_size)
    """
    img = Image.open(img_path).convert("RGB")
    img = img.resize((img_size, img_size), Image.BILINEAR)

    arr = np.asarray(img).astype(np.float32) / 255.0
    arr = (arr - IMAGENET_MEAN) / IMAGENET_STD
    arr = np.transpose(arr, (2, 0, 1))  # HWC -> CHW
    arr = np.expand_dims(arr, axis=0)   # Add batch dimension

    return arr


def load_calibration_images(calib_dir, img_size, max_images=500):
    """
    Load calibration images from folder.
    """
    calib_arrays = []

    supported_ext = [".jpg", ".jpeg", ".png", ".bmp", ".JPG", ".JPEG"]

    for file in os.listdir(calib_dir):
        if any(file.endswith(ext) for ext in supported_ext):
            img_path = os.path.join(calib_dir, file)
            calib_arrays.append(preprocess_image(img_path, img_size))

            if len(calib_arrays) >= max_images:
                break

    if len(calib_arrays) == 0:
        raise RuntimeError("No calibration images found!")

    return calib_arrays


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--onnx", required=True, help="Path to ONNX model")
    parser.add_argument("--mxq", required=True, help="Output MXQ path")
    parser.add_argument("--calib_dir", required=True, help="Calibration image folder")
    parser.add_argument("--img_size", type=int, default=240, help="Model input size")
    args = parser.parse_args()

    if not os.path.exists(args.onnx):
        raise FileNotFoundError("ONNX model not found!")

    if not os.path.exists(args.calib_dir):
        raise FileNotFoundError("Calibration folder not found!")

    print("Loading calibration images...")
    calib_data = load_calibration_images(args.calib_dir, args.img_size)

    print(f"Using {len(calib_data)} calibration images")
    print("Compiling ONNX -> MXQ ...")

    mxq_compile(
        model=args.onnx,
        save_path=args.mxq,
        calib_data_path=args.calib_dir,
        input_shapes=[[1, 3, args.img_size, args.img_size]],
        verbose=True,
    )

    if os.path.exists(args.mxq):
        size_mb = os.path.getsize(args.mxq) / (1024 * 1024)
        print(f"\n MXQ generated successfully: {args.mxq}")
        print(f"File size: {size_mb:.2f} MB")
    else:
        print(" MXQ generation failed!")


if __name__ == "__main__":
    main()